In [2]:
import numpy as np
from scipy.stats import lognorm, norm


def lognormal_samples(mean, cov, size, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    sigma_ln = np.sqrt(np.log(1 + cov ** 2))
    mu_ln    = np.log(mean) - 0.5 * sigma_ln ** 2
    return rng.lognormal(mean=mu_ln, sigma=sigma_ln, size=size)


def monte_carlo_failure(
    t_values,
    d_grown_array,
    x_F,
    N_samples=500_000,
    delta_mean=1.0,
    delta_cov=0.30,
    e_mean=1.0,
    e_cov=0.15,
    seed=None,
):
    """
    This runs monte carlo for g(t) = Delta - e * x_F * D_grown(t)

     given how the limit state is defined, it returns the probability of failure as an array
    """
    rng = np.random.default_rng(seed)

    Delta_samples = lognormal_samples(delta_mean, delta_cov, N_samples, rng)
    e_samples     = lognormal_samples(e_mean,     e_cov,     N_samples, rng)

    Pf = np.zeros(len(t_values), dtype=float)
    for i, t in enumerate(t_values):
        t_idx = int(np.round(t))
        if t_idx < 1:
            Pf[i] = 0.0
            continue
        t_idx = min(t_idx, len(d_grown_array))
        D_t = d_grown_array[t_idx - 1]

        g_vals = Delta_samples - e_samples * x_F * D_t
        Pf[i] = float(np.mean(g_vals < 0.0))

    return Pf


def Pf_to_beta(Pf):
    """β = -φ^-1(P_f), clipped to avoid inf"""
    Pf = np.asarray(Pf, dtype=float)
    Pf = np.clip(Pf, 1e-12, 1 - 1e-12)
    return -norm.ppf(Pf)


def beta_to_Pf(beta):
    return float(norm.cdf(-beta))
